# 03 Gold: star schema
```
               dim_date
                  |
dim_customer -- fact_orders -- dim_product
```

The fact table is dropped first because its foreign keys point at the dimension tables rebuilt below.

In [ ]:
%sql
DROP TABLE IF EXISTS olist.gold.fact_orders

In [ ]:
%sql
CREATE OR REPLACE TABLE olist.gold.dim_customer AS
SELECT customer_id, customer_city, customer_state
FROM olist.silver.customers

In [ ]:
%sql
CREATE OR REPLACE TABLE olist.gold.dim_product AS
SELECT product_id, product_category
FROM olist.silver.products

In [ ]:
%sql
CREATE OR REPLACE TABLE olist.gold.dim_date AS
SELECT DISTINCT
       order_date,
       YEAR(order_date)               AS year,
       MONTH(order_date)              AS month,
       DATE_TRUNC('month', order_date) AS month_start
FROM olist.silver.orders

Fact table: one row per item in an order.

In [ ]:
%sql
CREATE TABLE olist.gold.fact_orders AS
SELECT i.order_id,
       o.customer_id,
       i.product_id,
       o.order_date,
       i.price,
       i.freight_value,
       DATEDIFF(o.delivered_date, o.order_date) AS delivery_days
FROM olist.silver.order_items i
JOIN olist.silver.orders o ON i.order_id = o.order_id

In [ ]:
%sql
CREATE OR REPLACE TABLE olist.gold.payments AS
SELECT order_id, payment_type, payment_value
FROM olist.silver.payments

Check: sales by year, joining the fact table to a dimension.

In [ ]:
%sql
SELECT d.year, COUNT(DISTINCT f.order_id) AS orders, ROUND(SUM(f.price)) AS revenue
FROM olist.gold.fact_orders f
JOIN olist.gold.dim_date d ON f.order_date = d.order_date
GROUP BY d.year
ORDER BY d.year